In [ ]:
import pandas as pd
df = pd.read_csv("data/sample_transactions.csv")

df["date"] = pd.to_datetime(df["date"])
df["date"].dtype

df["date"].dt.month

pd.to_datetime(df["date"], errors="coerce")

test = pd.Series(["2026-01-01", "not a date"])
pd.to_datetime(test, errors="coerce")

test = pd.DataFrame({
    "date": ["2026-01-01", "not a date", "2026-01-03"],
    "amount": [100, 200, "abc"]
})

test["date"] = pd.to_datetime(test["date"], errors="coerce")
test["amount"] = pd.to_numeric(test["amount"], errors="coerce")
test

test["date"].isna()

bad = test["date"].isna() | test["amount"].isna()
bad

test[bad]     # the rejected rows
test[~bad]    # the good rows

test2 = pd.DataFrame({
    "description": ["Swiggy", "  ", None, "Netflix"],
    "amount": [-400, -100, -50, -649]
})

test2["description"] = test2["description"].str.strip()
test2

missing = test2["description"].isna() | (test2["description"] == "")
missing

test2.loc[missing, "description"] = "(no description)"
test2

test3 = pd.DataFrame({
    "date": ["2026-09-12", "2026-09-12", "2026-09-13"],
    "description": ["Swiggy Order", "Swiggy Order", "Swiggy Order"],
    "amount": [-450, -450, -450]
})

test3.duplicated(subset=["date", "description", "amount"])

is_dup = test3.duplicated(subset=["date", "description", "amount"])
test3[~is_dup]

is_dup.sum()



In [ ]:
import os
os.chdir(r"C:\Users\Hari Prasath\moneymap")


In [ ]:
from src.data_loader import load_transactions
from src.data_cleaning import clean_transactions

raw = load_transactions("data/sample_transactions.csv")
clean, rejected, summary = clean_transactions(raw)
summary


rules = {
    "swiggy": "Food Delivery",
    "zomato": "Food Delivery",
    "netflix": "Subscriptions",
}

desc = "Random Shop"

for keyword, category in rules.items():
    if keyword in desc.lower():
        print(category)

In [ ]:
def categorize(desc):
    for keyword, category in rules.items():
        if keyword in desc.lower():
            return category
    return "Uncategorized"

categorize("Swiggy Order")
categorize("Zomato Order")
categorize("random shop")

clean["category"] = clean["description"].apply(categorize)
clean.head(10)

clean["category"].value_counts()

rules = {
    "refund": "Refunds",
    "credit card payment": "Credit Card Payment",
    "own savings": "Internal Transfer",
    "salary": "Salary",
    "freelance": "Freelance Income",
    "rent": "Housing",
    "swiggy": "Food Delivery",
    "zomato": "Food Delivery",
    "netflix": "Subscriptions",
    "spotify": "Subscriptions",
    "groceries": "Groceries",
    "uber": "Transportation",
    "petrol": "Fuel",
    "electricity": "Utilities",
    "amazon": "Shopping",
    "myntra": "Shopping",
    "udemy": "Education",
    "pvr": "Entertainment",
    "pharmacy": "Healthcare",
    "clinic": "Healthcare",
    "indigo": "Travel",
}

clean["category"] = clean["description"].apply(categorize)
clean.head(10)

clean["category"].value_counts()

income = clean[clean["amount"] > 0]["amount"].sum()
expenses = clean[clean["amount"] < 0]["amount"].sum()
income, expenses

(income + expenses) / income * 100

In [ ]:
transfers = clean["category"] == "Internal Transfer"
refunds = clean["category"] == "Refunds"

# leave out transfers and refunds
money = clean[~transfers & ~refunds]

income = money[money["amount"] > 0]["amount"].sum()
expenses = money[money["amount"] < 0]["amount"].sum()

# refunds reduce expenses
expenses = expenses + clean[refunds]["amount"].sum()

income, expenses

(income + expenses) / income * 100

cc = clean["category"] == "Credit Card Payment"
cc_total = clean[cc]["amount"].sum()

expenses_without_cc = expenses - cc_total   # minus a negative = adds back
(income + expenses_without_cc) / income * 100

from src.analytics import calculate_summary

calculate_summary(clean)

calculate_summary(clean, count_card_payments=False)

In [1]:
import os
os.chdir(r"C:\Users\Hari Prasath\moneymap")
from src.data_loader import load_transactions
from src.data_cleaning import clean_transactions
from src.categorization import add_categories
from src.analytics import calculate_summary

raw = load_transactions("data/sample_transactions.csv")
clean, rejected, summary = clean_transactions(raw)
clean = add_categories(clean)

calculate_summary(clean)

spend = clean[(clean["amount"] < 0) & (clean["category"] != "Internal Transfer")]

by_cat = spend.groupby("category")["amount"].sum().abs().sort_values(ascending=False)
by_cat


import plotly.express as px
import pandas as pd

fig = px.bar(
    by_cat.reset_index(),
    x="amount",
    y="category",
    orientation="h",
    title="Spending by category",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

clean["month"] = clean["date"].dt.to_period("M").astype(str)
clean[["date", "month"]].head()

rows = []
for month, group in clean.groupby("month"):
    result = calculate_summary(group)
    rows.append({"month": month, "income": result["income"], "expenses": result["expenses"]})

monthly = pd.DataFrame(rows)
monthly

import plotly.express as px

fig = px.bar(
    monthly,
    x="month",
    y=["income", "expenses"],
    barmode="group",
    title="Monthly income vs expenses",
)
fig.show()

def pct_change(old, new):
    if old == 0:
        return None
    return (new - old) / old * 100

print(pct_change(1000, 1250))
print(pct_change(0, 500))
print(pct_change(1000, 800))

clean["month"] = clean["date"].dt.to_period("M").astype(str)

spend = clean[(clean["amount"] < 0) & (clean["category"] != "Internal Transfer")]

table = spend.pivot_table(
    index="category",
    columns="month",
    values="amount",
    aggfunc="sum",
).abs().fillna(0)

table

changes = {}
for category in table.index:
    old = table.loc[category, "2026-08"]
    new = table.loc[category, "2026-09"]
    changes[category] = pct_change(old, new)

pd.Series(changes)

from src.analytics import category_changes

category_changes(clean, "2026-08", "2026-09")


from src.data_loader import load_transactions
from src.data_cleaning import clean_transactions
from src.categorization import add_categories
from src.analytics import calculate_summary, category_changes

raw = load_transactions("data/sample_transactions.csv")
clean, rejected, summary = clean_transactions(raw)
clean = add_categories(clean)


spend = clean[(clean["amount"] < 0) & (clean["category"] != "Internal Transfer")]
by_cat = spend.groupby("category")["amount"].sum().abs().sort_values(ascending=False)

share = by_cat / by_cat.sum() * 100
share.round(1)
share.head(3).sum()

from src.analytics import category_shares

share, top3 = category_shares(clean)
print(top3)
share.round(1)

changes = category_changes(clean, "2026-07", "2026-08")

rising = changes[changes["change_pct"] > 20]
rising

for category, row in rising.iterrows():
    old = row["2026-07"]
    new = row["2026-08"]
    pct = row["change_pct"]
    print(f"{category} rose {pct:.1f}% (₹{old:,.0f} to ₹{new:,.0f})")

rising = changes[(changes["change_pct"] > 20) & ((changes["2026-08"] - changes["2026-07"]) >= 500)]

for category, row in rising.iterrows():
    print(f"{category} rose {row['change_pct']:.1f}% (₹{row['2026-07']:,.0f} to ₹{row['2026-08']:,.0f})")


from src.insights import rising_category_insights

for i in rising_category_insights(clean, "2026-07", "2026-08"):
    print(i)
    


import pandas as pd

bank = pd.DataFrame({
    "narration": ["Swiggy", "Salary", "Rent"],
    "debit": [450, None, 12000],
    "credit": [None, 45000, None],
})

bank["amount"] = bank["credit"].fillna(0) - bank["debit"].fillna(0)
bank

import io
from src.data_loader import load_transactions

text = """Date,Narration,Withdrawal,Deposit
2026-07-01,Swiggy,450,
2026-07-02,Salary,,45000
"""
load_transactions(io.StringIO(text))

load_transactions("data/sample_transactions.csv").shape

import pandas as pd

dates = pd.Series(["05/07/2026", "15/07/2026"])

pd.to_datetime(dates, errors="coerce")

pd.to_datetime(dates, errors="coerce", dayfirst=True)
pd.to_datetime(dates, errors="coerce")

def parse_dates(series):
    for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"]:
        parsed = pd.to_datetime(series, format=fmt, errors="coerce")
        if parsed.notna().all():
            return parsed, fmt
    return pd.to_datetime(series, errors="coerce"), "unknown"

print(parse_dates(pd.Series(["2026-07-01", "2026-07-15"])))
print(parse_dates(pd.Series(["05/07/2026", "15/07/2026"])))
print(parse_dates(pd.Series(["05-07-2026", "15-07-2026"])))

import io
from src.data_loader import load_transactions
from src.data_cleaning import clean_transactions

text = """Date,Narration,Withdrawal,Deposit
05/07/2026,Swiggy,450,
15/07/2026,Salary,,45000
not a date,Rent,12000,
"""
clean, rejected, summary = clean_transactions(load_transactions(io.StringIO(text)))
print(summary)
clean

from src.data_loader import load_transactions
from src.data_cleaning import clean_transactions
from src.categorization import add_categories
from src.analytics import calculate_summary, category_changes

raw = load_transactions("data/sample_transactions.csv")
clean, rejected, summary = clean_transactions(raw)
clean = add_categories(clean)

from src.analytics import category_changes

changes = category_changes(clean, "2026-08", "2026-09")

new_spend = changes[(changes["2026-08"] == 0) & (changes["2026-09"] >= 1000)]
new_spend

from src.insights import new_spending_insights

for i in new_spending_insights(clean, "2026-08", "2026-09"):
    print(i)


spend = clean[clean["amount"] < 0]

groups = spend.groupby("description")["amount"].agg(["count", "mean", "std"])
groups[groups["count"] >= 3]

groups["wobble"] = groups["std"] / groups["mean"].abs()
groups[groups["count"] >= 3].sort_values("wobble").round(3)

netflix = spend[spend["description"] == "Netflix Subscription"].sort_values("date")
netflix["gap_days"] = netflix["date"].diff().dt.days
netflix[["date", "gap_days"]]

swiggy = spend[spend["description"] == "Swiggy Order"].sort_values("date")
swiggy["gap_days"] = swiggy["date"].diff().dt.days
swiggy[["date", "gap_days"]]

def gap_stats(g):
    gaps = g["date"].sort_values().diff().dt.days.dropna()
    return pd.Series({
        "avg_gap_days": gaps.mean(),
        "gap_wobble": gaps.std() / gaps.mean(),
    })

gaps = spend.groupby("description")[["date"]].apply(gap_stats)
gaps.round(2).sort_values("gap_wobble")

stats = groups.join(gaps)

candidates = stats[
    (stats["count"] >= 3)
    & (stats["wobble"] <= 0.1)
    & (stats["gap_wobble"] <= 0.2)
]
candidates.round(3)

from src.recurring_payments import find_recurring

find_recurring(clean)


import pandas as pd

clean["month"] = clean["date"].dt.to_period("M").astype(str)

rows = []
for month, group in clean.groupby("month"):
    r = calculate_summary(group)
    rows.append({"month": month, "savings_rate": r["savings_rate"]})

trend = pd.DataFrame(rows)
trend

trend["change_points"] = trend["savings_rate"].diff()
trend.round(1)

from src.analytics import monthly_summary

monthly_summary(clean).round(1)

last_date = clean["date"].max()
month_end = last_date + pd.offsets.MonthEnd(0)

last_date, month_end

from src.analytics import is_incomplete_last_month

print(is_incomplete_last_month(clean))

early = clean[clean["date"] <= "2026-09-10"]
print(is_incomplete_last_month(early))


spend = clean[(clean["amount"] < 0) & (clean["category"] != "Internal Transfer")]

top = spend.groupby("description")["amount"].agg(["sum", "count"])
top["sum"] = top["sum"].abs()
top = top.sort_values("sum", ascending=False).head(5)
top

from src.analytics import top_merchants

top_merchants(clean).round(1)

25.0
None
-20.0
67.38166938461076
Credit Card Payment rose 25.0% (₹6,000 to ₹7,500)
Food Delivery rose 37.5% (₹1,360 to ₹1,870)
Shopping rose 39.1% (₹2,300 to ₹3,200)
Transportation rose 24.0% (₹250 to ₹310)
Credit Card Payment rose 25.0% (₹6,000 to ₹7,500)
Food Delivery rose 37.5% (₹1,360 to ₹1,870)
Shopping rose 39.1% (₹2,300 to ₹3,200)
{'title': 'Credit Card Payment spending increased', 'observation': 'Credit Card Payment rose 25.0% from 2026-07 to 2026-08.', 'evidence': '₹6,000 in 2026-07, ₹7,500 in 2026-08.', 'caveat': 'Card payments stand in for card purchases that are not in the data.'}
{'title': 'Food Delivery spending increased', 'observation': 'Food Delivery rose 37.5% from 2026-07 to 2026-08.', 'evidence': '₹1,360 in 2026-07, ₹1,870 in 2026-08.', 'caveat': ''}
{'title': 'Shopping spending increased', 'observation': 'Shopping rose 39.1% from 2026-07 to 2026-08.', 'evidence': '₹2,300 in 2026-07, ₹3,200 in 2026-08.', 'caveat': ''}
(0   2026-07-01
1   2026-07-15
dtype: datetime6

,total,payments,share_pct
description,,,
Rent - Landlord Transfer,36000,3,36.0
HDFC Credit Card Payment,21500,3,21.5
Big Bazaar Groceries,5900,3,5.9
Amazon Shopping,5050,2,5.1
IndiGo Flight Ticket,4800,1,4.8
